# Day 20 — Practice Session · **SOLUTIONS**
### Outliers, Matplotlib & Seaborn · Python for Data Science

**Prepared by Srinivasa Sai Chava**  ·  Boston University

---

> **Instructor copy.** Every question is followed by the answer and the reasoning.
> The student copy (`Day20_Practice_Questions.ipynb`) is identical minus the answer blocks.

| Part | Focus | Questions |
|---|---|---|
| A | Predict the output | 8 |
| B | Spot & fix the bug | 4 |
| C | Write the code | 5 |
| D | Challenge | 2 |

---
## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter("ignore")

np.random.seed(42)
sales = pd.DataFrame({
    "rep":    [f"R{i:02d}" for i in range(30)],
    "region": np.random.choice(["North", "South", "East"], 30),
    "visits": np.random.randint(5, 25, 30),
    "revenue": np.r_[np.random.normal(40000, 6000, 27), [185000, 192000, 205000]].round(0),
})
print("rows:", len(sales))
sales.head()

---
## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter("ignore")

np.random.seed(42)
sales = pd.DataFrame({
    "rep":    [f"R{i:02d}" for i in range(30)],
    "region": np.random.choice(["North", "South", "East"], 30),
    "visits": np.random.randint(5, 25, 30),
    "revenue": np.r_[np.random.normal(40000, 6000, 27), [185000, 192000, 205000]].round(0),
})
print("rows:", len(sales))
sales.head()

---
# Part A — Predict the Output  *(8 min)*

---
# Part A — Predict the Output

**Teaching note:** A3 (the z-score blind spot) and A7 (barplot showing the mean) are the two
that change behaviour.

### A1. What are the bounds?

In [ ]:
d = pd.Series([12., 15., 14., 13., 16., 15., 14., 95., 13.])
q1, q3 = d.quantile([.25, .75])
iqr = q3 - q1
print(q1, q3, iqr)
print(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

*Your prediction:*  

> ### ✅ Answer A1
> ```
> 13.0 15.0 2.0
> 10.0 18.0
> ```
> **Why:** the quartiles sit at 13 and 15, so the IQR — the spread of the middle half — is
> just 2.0. The bounds are 13 − 3 = 10 and 15 + 3 = 18.
>
> **The 95 did not move these bounds at all.** Quartiles are positions, so a single extreme
> value cannot drag them. That is exactly what makes the IQR rule robust.

### A2. Which values are flagged?

In [ ]:
d = pd.Series([12., 15., 14., 13., 16., 15., 14., 95., 13.])
lower, upper = 10.0, 18.0
print(d[(d < lower) | (d > upper)].tolist())
print(len(d[(d >= lower) & (d <= upper)]))

*Your prediction:*  

> ### ✅ Answer A2
> ```
> [95.0]
> 8
> ```
> **Why:** only the 95 falls outside. Eight of nine values survive.
>
> Note this is just Day 12's boolean masking with `|` and `&`, and the brackets around each
> condition are still required.

### A3. Does the z-score find it?

In [ ]:
d = pd.Series([12., 15., 14., 13., 16., 15., 14., 95., 13.])
z = (d - d.mean()) / d.std()
print(round(d.mean(), 2), round(d.std(), 2))
print(round(z.abs().max(), 2))

*Your prediction:*  

> ### ✅ Answer A3
> ```
> 23.0 27.03
> 2.66
> ```
> **No — and this is the most important answer in the notebook.** The usual z-score
> threshold is 3, and the largest |z| here is **2.66**. The method fails to flag the very
> value that is distorting the dataset.
>
> **Why:** the z-score is built from the mean and the standard deviation. The 95 pushed the
> mean from 14 up to 23, and the std from 1.3 up to 27 — so it inflated the yardstick it was
> about to be measured against.
>
> Against the clean statistics (mean 14.0, std 1.31) its z-score is **61.9**.
>
> **Use the IQR rule by default.** Use z-scores only when you already know the data is
> roughly normal and free of gross errors.

### A4. What does clip do?

In [ ]:
d = pd.Series([12., 15., 14., 13., 16., 15., 14., 95., 13.])
capped = d.clip(10.0, 18.0)
print(capped.tolist())
print(len(d), len(capped))

*Your prediction:*  

> ### ✅ Answer A4
> ```
> [12.0, 15.0, 14.0, 13.0, 16.0, 15.0, 14.0, 18.0, 13.0]
> 9 9
> ```
> **Why:** `clip` pulls anything outside the range to the nearest boundary. The 95 becomes
> 18.0 and everything else is already inside, so it is untouched.
>
> **The row count does not change** — that is the point of capping rather than removing. The
> record stays in your dataset; only the extreme value is tamed. The cost is that 18.0 is a
> number you invented.

### A5. What do these four datasets have in common?

In [ ]:
x = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
y1 = [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]
y2 = [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]
for y in [y1, y2]:
    s = pd.Series(y)
    print(round(s.mean(), 2), round(s.std(), 2), round(pd.Series(x).corr(s), 3))

*Your prediction:*  

> ### ✅ Answer A5
> ```
> 7.5 2.03 0.816
> 7.5 2.03 0.816
> ```
> **Identical, to three decimals.** These are datasets I and II of Anscombe's quartet — the
> first is roughly linear, the second is a clean curve.
>
> Mean, standard deviation and correlation are all blind to the difference. **Only plotting
> reveals it**, which is why plotting is a step in the process rather than a presentation
> afterthought.

### A6. What does subplots return?

In [ ]:
fig, ax = plt.subplots()
print(type(ax).__name__)
fig2, axes = plt.subplots(2, 2)
print(axes.shape)
print(type(axes).__name__)
plt.close("all")

*Your prediction:*  

> ### ✅ Answer A6
> ```
> Axes
> (2, 2)
> ndarray
> ```
> **Why:** with no arguments you get one `Axes`. With `(2, 2)` you get a **NumPy array** of
> them, shaped `(2, 2)`.
>
> So you index it exactly as Day 12 taught: `axes[0, 1]` is the top-right plot. For a single
> row, `plt.subplots(1, 3)` gives a 1-D array indexed `axes[0]`, `axes[1]`, `axes[2]`.
>
> This is why `fig, ax = plt.subplots()` is the habit to build — the moment you want more
> than one chart, bare `plt.plot()` has nowhere to put it.

### A7. What height is bar A?

In [ ]:
small = pd.DataFrame({"g": ["A", "A", "B"], "v": [10, 20, 30]})
print(small.groupby("g")["v"].sum().to_dict())
print(small.groupby("g")["v"].mean().to_dict())
# sns.barplot(data=small, x="g", y="v")  -> what would A show?

*Your prediction:*  

> ### ✅ Answer A7
> ```
> {'A': 30, 'B': 30}
> {'A': 15.0, 'B': 30.0}
> ```
> **`sns.barplot` would show A at 15, not 30** — it plots the **mean** by default, not the
> sum.
>
> So a chart that looks like "B sells twice as much as A" is really saying "B's average sale
> is twice A's average sale", while their totals are **identical**. Any reader will assume
> bars are totals.
>
> **Two fixes:** `estimator="sum"`, or group the data yourself first so you can see the
> numbers before they are drawn. Seaborn also adds a confidence interval line by default —
> `errorbar=None` removes it.

### A8. Does this work?

In [ ]:
df = pd.DataFrame({"a": [1, 2, 3], "b": [2, 4, 7], "name": ["x", "y", "z"]})
print(df.corr())

*Your prediction:*  

> ### ✅ Answer A8
> ```
> ValueError
> ```
> **Why:** `corr()` cannot correlate a text column, and recent pandas raises rather than
> silently dropping it.
>
> **The fix:** `df.corr(numeric_only=True)` — which gives the 2×2 matrix of `a` against `b`,
> with r = 0.993.
>
> This matters because `sns.heatmap(df.corr())` is a standard EDA move, and almost every
> real DataFrame has at least one text column in it.

---
# Part B — Spot & Fix the Bug  *(7 min)*

---
# Part B — Spot & Fix the Bug

**Teaching note:** B1 is a correctness bug; B2–B4 are communication bugs. Both kinds lose
marks in the project.

### B1. This should find the unusual revenues.

In [ ]:
z = (sales["revenue"] - sales["revenue"].mean()) / sales["revenue"].std()
outliers = sales[z.abs() > 3]
print(f"found {len(outliers)} outliers")

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B1
> **Symptom:** finds only **1 outlier** of the three. The reps at 185,000 and 192,000 score
> z = 2.73 and 2.88 — both under the threshold — so only the 205,000 is caught.
> **Cause:** the three extreme values together dragged the mean from about 40,000 up to
> **55,745** and the standard deviation up to **47,349**. Measured against that inflated
> yardstick, two of the three outliers look ordinary.
>
> **The more outliers there are, the worse the z-score method performs** — which is precisely
> backwards from what you want.

In [ ]:
z = (sales["revenue"] - sales["revenue"].mean()) / sales["revenue"].std()
print("z-score method :", (z.abs() > 3).sum(), "of the 3 actual outliers")
print(f"  mean {sales['revenue'].mean():,.0f}  std {sales['revenue'].std():,.0f}")
print("  z of the three extremes:", [round(v, 2) for v in z.tail(3)])
print("  -> 2.73 and 2.88 are under the threshold of 3, so they slip through")

# THE FIX - the IQR rule
q1, q3 = sales["revenue"].quantile([.25, .75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
found = sales[(sales["revenue"] < lo) | (sales["revenue"] > hi)]

print()
print("IQR method     :", len(found), "outliers")
print(f"  bounds {lo:,.0f} to {hi:,.0f}")
print(found[["rep", "region", "revenue"]])

# Quartiles are POSITIONS, so three extreme values cannot move them.

### B2. This should show two charts side by side.

In [ ]:
plt.plot(sales["visits"])
plt.plot(sales["revenue"])
plt.title("Visits and revenue")
plt.show()

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B2
> **Symptom:** both series land on the **same** axes, and because revenue is in the tens of
> thousands while visits are under 25, the visits line is flattened to a straight line along
> the bottom. The chart is unreadable.
> **Cause:** bare `plt.plot()` draws on whatever figure is current. There is no second
> figure, so everything stacks.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(sales["visits"], color="#3E7CB1")
axes[0].set_title("Visits per rep")
axes[0].set_xlabel("rep index"); axes[0].set_ylabel("visits")

axes[1].plot(sales["revenue"], color="#2FA98C")
axes[1].set_title("Revenue per rep")
axes[1].set_xlabel("rep index"); axes[1].set_ylabel("revenue (Rs)")

fig.tight_layout()
plt.show()

# Two quantities on wildly different scales need two axes - or a twin axis,
# which is usually harder to read than two charts.

### B3. This should show total revenue per region.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.barplot(data=sales, x="region", y="revenue", ax=ax)
ax.set_title("Total revenue by region")
plt.show()

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B3
> **Symptom:** the title says *total* but the bars show the **mean**. There is also a
> confidence interval line on each bar that nobody asked for.
> **Cause:** `sns.barplot`'s default estimator is the mean.

In [ ]:
print("the actual totals:")
print(sales.groupby("region")["revenue"].sum().round(0))
print()
print("what the chart was showing (means):")
print(sales.groupby("region")["revenue"].mean().round(0))

# THE FIX - group first, so you can see the numbers before drawing them
totals = sales.groupby("region", as_index=False)["revenue"].sum()

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.barplot(data=totals.sort_values("revenue", ascending=False),
            x="region", y="revenue", ax=ax)
ax.set_title("Total revenue by region")
ax.set_xlabel("Region"); ax.set_ylabel("Total revenue (Rs)")
plt.show()

# Sorting the bars is the other half of the fix - an unsorted bar chart
# makes the reader do work they should not have to.

### B4. This heatmap should show the correlations.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(sales.corr(), annot=True, ax=ax)
plt.show()

*Error:* 

*Your fix:*

> ### ✅ Answer B4
> **Error:** `ValueError` — `sales` contains the text columns `rep` and `region`, and
> `corr()` cannot correlate text.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(sales.corr(numeric_only=True), annot=True, cmap="Blues",
            fmt=".2f", ax=ax)
ax.set_title("Correlation: visits vs revenue")
plt.show()

print(sales.corr(numeric_only=True).round(3))

# numeric_only=True keeps just the numeric columns.
# annot=True writes the numbers in the cells, fmt=".2f" rounds them,
# and a title says what the reader is looking at.

---
# Part C — Write the Code  *(12 min)*

Label every chart.

---
# Part C — Write the Code

**Teaching note:** C5 is the template to make them keep — it is tomorrow's project opening
move.

### C1. Detect and inspect
Using the IQR rule on `revenue`, print the bounds, how many outliers there are, and the
actual rows.

In [ ]:
# your code here

In [ ]:
q1, q3 = sales["revenue"].quantile([.25, .75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

odd = sales[(sales["revenue"] < lo) | (sales["revenue"] > hi)]

print(f"Q1 {q1:,.0f}   Q3 {q3:,.0f}   IQR {iqr:,.0f}")
print(f"bounds: below {lo:,.0f} or above {hi:,.0f}")
print(f"{len(odd)} outliers of {len(sales)} rows")
print()
print(odd[["rep", "region", "visits", "revenue"]])

# PRINTING THE ROWS is the step that matters. Looking at them, these three
# reps have ordinary visit counts but revenue about five times the rest -
# which is a question for the sales manager, not something to delete.

### C2. Compare the treatments
For `revenue`, print the mean, std and row count under three strategies: keep everything,
remove the outliers, and cap them at the IQR bounds.

In [ ]:
# your code here

In [ ]:
q1, q3 = sales["revenue"].quantile([.25, .75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

r = sales["revenue"]
rows = []
for label, v in [("keep",   r),
                 ("remove", r[(r >= lo) & (r <= hi)]),
                 ("cap",    r.clip(lo, hi))]:
    rows.append({"strategy": label, "mean": round(v.mean()), "std": round(v.std()), "n": len(v)})

print(pd.DataFrame(rows).set_index("strategy"))

# Keeping them roughly doubles the std. Removing and capping give similar
# means, but capping keeps all 30 rows - the three reps stay in the analysis.
#
# Which is right depends on the question. For "typical rep performance",
# cap or remove. For "who are our top performers", keeping them IS the point.

### C3. A labelled histogram
Plot the distribution of `revenue` with 20 bins. Label both axes, add a title, and draw a
vertical line at the upper IQR bound.

In [ ]:
# your code here

In [ ]:
q1, q3 = sales["revenue"].quantile([.25, .75])
hi = q3 + 1.5 * (q3 - q1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sales["revenue"], bins=20, color="#3E7CB1", edgecolor="white")
ax.axvline(hi, color="#E8674C", linestyle="--", linewidth=2,
           label=f"upper IQR bound ({hi:,.0f})")

ax.set_title("Distribution of revenue per rep")
ax.set_xlabel("Revenue (Rs)")
ax.set_ylabel("Number of reps")
ax.legend()
plt.show()

# axvline is the quickest way to mark a threshold on any chart.
# The gap between the main cluster and the three far-right bars is exactly
# what a summary statistic would have hidden.

### C4. A grouped comparison
Draw a boxplot of `revenue` by `region`, and say what the dots above the whiskers are.

In [ ]:
# your code here

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=sales, x="region", y="revenue", ax=ax)

ax.set_title("Revenue by region (dots beyond the whiskers are outliers)")
ax.set_xlabel("Region")
ax.set_ylabel("Revenue (Rs)")
plt.show()

# THE DOTS ARE THE OUTLIERS. The whiskers are drawn at exactly the
# 1.5 x IQR bounds from section 1 - so a boxplot is the detection rule
# of section 1 and the picture of section 3, in one chart.
#
# Seaborn computes the bounds PER GROUP, so each region gets its own.
print(sales.groupby("region")["revenue"].describe()[["25%", "50%", "75%"]].round(0))

### C5. A 2×2 overview
Build a single figure with four panels: a histogram of revenue, a boxplot by region, a
scatter of visits against revenue coloured by region, and a correlation heatmap. Title every
panel.

In [ ]:
# your code here

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

sns.histplot(data=sales, x="revenue", bins=20, ax=axes[0, 0])
axes[0, 0].set_title("Revenue: distribution")
axes[0, 0].set_xlabel("Revenue (Rs)")

sns.boxplot(data=sales, x="region", y="revenue", ax=axes[0, 1])
axes[0, 1].set_title("Revenue by region")

sns.scatterplot(data=sales, x="visits", y="revenue", hue="region",
                alpha=0.8, ax=axes[1, 0])
axes[1, 0].set_title("Visits vs revenue")

sns.heatmap(sales.corr(numeric_only=True), annot=True, cmap="Blues",
            fmt=".2f", ax=axes[1, 1])
axes[1, 1].set_title("Correlations")

fig.suptitle("Sales team: first look", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig("overview.png", dpi=150, bbox_inches="tight")
plt.show()

# KEEP THIS TEMPLATE. Four panels is a complete first look at any dataset,
# and it is tomorrow's project in miniature.
#
# Note the scatter: the three high-revenue reps do NOT have unusually high
# visit counts, which is the interesting part - and no summary statistic
# would have told you.

---
# Part D — Challenge  *(3 min, or take home)*

---
# Part D — Challenge

**Teaching note:** D1 is the judgement exercise — the outlier you should KEEP. D2 is the
communication exercise.

### D1. The outlier you should keep
Three reps have very high revenue. Investigate: are they errors, or are they the finding?
Use the data to argue one way or the other, then say what you would do.

In [ ]:
# your code here

In [ ]:
q1, q3 = sales["revenue"].quantile([.25, .75])
iqr = q3 - q1
hi = q3 + 1.5 * iqr
top = sales[sales["revenue"] > hi]

print("the three flagged reps:")
print(top[["rep", "region", "visits", "revenue"]])
print()
print("everyone else:")
print(sales[sales["revenue"] <= hi][["visits", "revenue"]].describe().round(0).loc[["mean", "min", "max"]])

# ---- the argument
print()
print("Evidence they are REAL, not errors:")
print(f"  - their visit counts ({top['visits'].tolist()}) are in the normal range")
print(f"    (everyone else: {sales[sales['revenue'] <= hi]['visits'].min()}"
      f"-{sales[sales['revenue'] <= hi]['visits'].max()})")
print("  - three separate reps, not one - a typo is usually a single row")
print("  - the values are not round numbers or sentinels like 999999")
print("  - they span more than one region")

print()
print("WHAT I WOULD DO: keep them, and report them SEPARATELY.")
print("  'Typical rep revenue is about 40,000 (27 reps).'")
print("  'Three reps are at 185,000-205,000 and warrant investigation.'")
print()
print("Deleting them would delete the most interesting fact in the dataset.")
print("Capping them would hide it. The right move is to SPLIT the reporting.")

# This is the judgement the IQR rule cannot make for you. It found the
# candidates; deciding what they MEAN needed you to look at the other columns.

### D2. Make a chart that cannot be misread
Draw the single chart you would put in front of a sales director to explain the revenue
distribution. Then write one sentence saying what it shows.

In [ ]:
# your code here

In [ ]:
q1, q3 = sales["revenue"].quantile([.25, .75])
hi = q3 + 1.5 * (q3 - q1)
typical = sales[sales["revenue"] <= hi]

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.hist(typical["revenue"], bins=15, color="#3E7CB1",
        edgecolor="white", label=f"typical reps (n={len(typical)})")
ax.hist(sales[sales["revenue"] > hi]["revenue"], bins=15, color="#E8674C",
        edgecolor="white", label=f"exceptional reps (n={len(sales) - len(typical)})")

ax.set_title("Revenue per sales rep: 27 typical, 3 exceptional", fontsize=13)
ax.set_xlabel("Revenue (Rs)")
ax.set_ylabel("Number of reps")
ax.legend()
fig.tight_layout()
plt.show()

print("ONE SENTENCE:")
print(f"  Most reps earn between {typical['revenue'].min():,.0f} and "
      f"{typical['revenue'].max():,.0f}, but three earn roughly five times that,")
print("  and the gap between the two groups is large enough that averaging them")
print("  together would describe nobody.")

# WHY THIS CHART WORKS:
#   - the two groups are coloured differently, so the split is the message
#   - the legend states the counts, so nobody has to guess
#   - the title says the finding, not just the variable name
#   - the x-axis has units
#
# Compare with a single bar showing "average revenue: 55,000" - a number
# that describes neither group.

---
## Done? Self-check

- [ ] I can compute the 1.5 × IQR bounds from memory
- [ ] I know why a z-score can miss its own outlier
- [ ] I printed the outlier rows before deciding anything
- [ ] I know the four treatments and what each costs
- [ ] I used `fig, ax = plt.subplots()` every time
- [ ] Every chart I made has labelled axes and a title
- [ ] I know `sns.barplot` shows the mean, not the sum

### Homework
1. Find the outliers in a dataset by IQR and by z-score, and explain any disagreement.
2. Rebuild the 2×2 overview grid for a dataset of your own.
3. Take one chart and improve it until a classmate reads it correctly.

### Next class — Day 21: Mini project
A full EDA on a real dataset. Load, clean, explore, visualise and write up — everything from
Days 15 to 20, end to end.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*

---
## Wrap-up — running the last 5 minutes

Three cold-call questions:

1. *"Why use IQR rather than z-scores?"* → the outlier inflates the z-score's own yardstick.
2. *"`sns.barplot` — mean or sum?"* → mean, with a confidence interval.
3. *"What are the dots on a boxplot?"* → the points beyond the 1.5 × IQR whiskers.

**Common misconceptions to watch for today**

| Misconception | Correction |
|---|---|
| "Outliers should be removed" | They may be the finding. Look first |
| "Z-scores are the standard method" | They fail on exactly the data you need them for |
| "Capping is the same as removing" | Capping keeps the row and invents a value |
| "Summary statistics describe the data" | Anscombe: four shapes, identical statistics |
| "`plt.plot()` is fine" | Not once you need two charts |
| "A bar chart shows totals" | `sns.barplot` shows means by default |
| "`df.corr()` just works" | It raises on text columns — `numeric_only=True` |

**B1 is worth running live.** The z-score method catches only **one** of the three obvious
outliers: the other two score 2.73 and 2.88, just under the threshold, because together they
dragged the mean to 55,745 and the standard deviation to 47,349. The IQR rule finds all
three. Students taught z-scores as *the* method find this genuinely surprising, and it is
the clearest possible justification for using quartiles instead.

**D1 is the judgement exercise.** The three high-revenue reps have *normal visit counts*,
which is evidence they are real rather than typos — and that argument comes from looking at
the other columns, not from any outlier test. The right answer is to keep them and report
them separately, because averaging the two groups together describes nobody.

**On pace:** this session is compressed by design. If something had to be cut, cut depth on
matplotlib styling — students can look up colours and markers. Do not cut the z-score
failure or the barplot default; those are the two that cause wrong work.

**Homework given:** IQR versus z-score with an explanation; the 2×2 grid on their own data;
one chart improved until a classmate reads it correctly.

**Next session:** Day 21 — the mini project. A real dataset from raw file to written
findings, using everything from Days 15 to 20. Today's 2×2 overview grid is the opening move.